## ETL: Registro de Atendimentos da Defesa Civil do Recife

### Importação das bibliotecas

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

### Conexão com o banco

In [2]:
load_dotenv("../.env")

usuario = os.getenv("DB_USER")
senha = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
porta = os.getenv("DB_PORT")
banco = os.getenv("DB_NAME")

database_url = f"postgresql+psycopg2://{usuario}:{senha}@{host}:{porta}/{banco}"
engine = create_engine(database_url)

with engine.connect():
    print("Conexão criada com sucesso.")

Conexão criada com sucesso.


### Extração dos dados

In [3]:
df_2022 = pd.read_csv("data/atendimentos-2022.csv", sep=";", dtype=str, encoding="utf-8")
df_2023 = pd.read_csv("data/atendimentos-2023.csv", sep=";", dtype=str, encoding="utf-8")
df_2024 = pd.read_csv("data/atendimentos-2024.csv", sep=";", dtype=str, encoding="utf-8")

print(df_2022.shape)
print(df_2023.shape)
print(df_2024.shape)

(90425, 16)
(83164, 16)
(80173, 16)


### Transformação dos dados

#### Diagnóstico do problema
Embora as 3 bases tenham as mesmas 16 colunas no cabeçalho, o arquivo de 2022 não possui valores para a coluna **Regional**. Com isso, os dados aparecem deslocados uma coluna para a esquerda: o ano sob "Regional", o mês sob "Ano", e assim por diante. Além disso, o endereço vem quebrado em dois campos, com o nome da rua separado do número anonimizado. A partir da coluna **Bairro**, os dados voltam a ficar alinhados. A comparação abaixo, entre as primeiras linhas de 2022 e de 2023, evidencia o problema.

In [4]:
# exibe as primeiras linhas dos dataframes para verificação
display(df_2022.head(3))
display(df_2023.head(3))

,Regional,Ano,Mês,Data,Ocorrencia,Solicitacao,Endereco,Bairro,Localidade,Grau_de_Risco,Data_da_Acao,Tipo_da_Acao,Quantidade,Altura,Largura,Area_m
0,2022,ABRIL,2022-04-19,Imveis com Danos / Risco,Vistoria em Imveis,"""Rua Jose Moreira Reis","{})!""",Bongi,Sem Localidade,NaN,2022-04-19,Analise Visual Estrutural,NaN,NaN,NaN,0
1,2022,ABRIL,2022-04-19,NaN,NaN,"""Avenida Dantas Barreto","`=-[""",Santo Antonio,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0
2,2022,ABRIL,2022-04-19,NaN,NaN,"""Rua Velha","?_$.""",Boa Vista,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0


,Regional,Ano,Mês,Data,Ocorrencia,Solicitacao,Endereco,Bairro,Localidade,Grau_de_Risco,Data_da_Acao,Tipo_da_Acao,Quantidade,Altura,Largura,Area_m
0,Oeste,2023,ABRIL,2023-04-27,NaN,NaN,"Rua Monte Pascoal, ""#*9",Cohab,Jardim Monte Verde,NaN,2023-03-27,ATENDIMENTO SOCIAL,NaN,NaN,NaN,0
1,Oeste,2023,ABRIL,2023-04-28,NaN,Monitoramento,"Rua Barra do Sagi, $!>=",Varzea,Brega e Chique,R3,2023-04-28,NaN,NaN,NaN,NaN,0
2,Oeste,2023,ABRIL,2023-04-28,NaN,Monitoramento,"Rua Jose Antonio da Costa Filho, >#'\",Varzea,Jardim Teresopolis,R3,2023-04-28,NaN,NaN,NaN,NaN,0


#### Correção do problema
A função **consertar_2022** resolve esses problemas. Primeiro, junta os dois campos do endereço com uma vírgula e remove as aspas que sobraram da quebra. Em seguida, atribui a cada coluna deslocada o valor que estava na anterior, e cria a coluna **Regional** vazia, já que essa informação não existe no arquivo de 2022.

In [5]:
def consertar_2022(df):
    # refaz o endereço, que veio quebrado em dois campos
    rua = df["Solicitacao"].fillna("").str.lstrip('"')
    numero = df["Endereco"].fillna("").str.rstrip('"')
    endereco = rua + "," + numero
    
    # realinha as colunas deslocadas
    colunas_corrigidas = pd.DataFrame({
        "Regional": pd.NA,
        "Ano": df["Regional"],
        "Mês": df["Ano"],
        "Data": df["Mês"],
        "Ocorrencia": df["Data"],
        "Solicitacao": df["Ocorrencia"],
        "Endereco": endereco
    })

    colunas_alinhadas = ["Bairro", "Localidade", "Grau_de_Risco", "Data_da_Acao", "Tipo_da_Acao", "Quantidade", "Altura", "Largura", "Area_m"]

    return pd.concat([colunas_corrigidas, df[colunas_alinhadas]], axis=1) # axis=1 concatena lado a lado

df_2022 = consertar_2022(df_2022)
df_2022.head(3)

,Regional,Ano,Mês,Data,Ocorrencia,Solicitacao,Endereco,Bairro,Localidade,Grau_de_Risco,Data_da_Acao,Tipo_da_Acao,Quantidade,Altura,Largura,Area_m
0,<NA>,2022,ABRIL,2022-04-19,Imveis com Danos / Risco,Vistoria em Imveis,"Rua Jose Moreira Reis, {})!",Bongi,Sem Localidade,NaN,2022-04-19,Analise Visual Estrutural,NaN,NaN,NaN,0
1,<NA>,2022,ABRIL,2022-04-19,NaN,NaN,"Avenida Dantas Barreto, `=-[",Santo Antonio,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0
2,<NA>,2022,ABRIL,2022-04-19,NaN,NaN,"Rua Velha, ?_$.",Boa Vista,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0


#### Consolidação das bases
Cada base recebe a coluna **ano_origem**, que indica de qual arquivo a linha veio. Isso é necessário porque os arquivos são organizados pela data da ação, e alguns atendimentos têm data de registro em outro ano. Depois, as três bases são empilhadas numa só. O total esperado é de 253.762 linhas.

In [6]:
df_2022["ano_origem"] = "2022"
df_2023["ano_origem"] = "2023"
df_2024["ano_origem"] = "2024"

df = pd.concat([df_2022, df_2023, df_2024], ignore_index=True)
print(f"Total de linhas da base consolidada: {len(df)}")

Total de linhas da base consolidada: 253762


#### Renomeação das colunas
Os nomes das colunas são padronizados em letras minúsculas, sem acentos e com palavras separadas por *underscore*. Os conectivos também são removidos, seguindo o mesmo padrão dos nomes das dimensões do modelo. Além disso, a coluna **Data** é renomeada para **data_registro**, para diferenciá-la da data da ação, já que cada atendimento possui as duas. Isso evita erros de digitação no código e mantém os nomes iguais em todas as etapas do projeto, incluindo as consultas em SQL.

In [7]:
df = df.rename(columns={
    "Regional": "regional",
    "Ano": "ano",
    "Mês": "mes",
    "Data": "data_registro",
    "Ocorrencia": "ocorrencia",
    "Solicitacao": "solicitacao",
    "Endereco": "endereco",
    "Bairro": "bairro",
    "Localidade": "localidade",
    "Grau_de_Risco": "grau_risco",
    "Data_da_Acao": "data_acao",
    "Tipo_da_Acao": "tipo_acao",
    "Quantidade": "quantidade",
    "Altura": "altura",
    "Largura": "largura",
    "Area_m": "area_m"    
})

df.head(3)

,regional,ano,mes,data_registro,ocorrencia,solicitacao,endereco,bairro,localidade,grau_risco,data_acao,tipo_acao,quantidade,altura,largura,area_m,ano_origem
0,<NA>,2022,ABRIL,2022-04-19,Imveis com Danos / Risco,Vistoria em Imveis,"Rua Jose Moreira Reis, {})!",Bongi,Sem Localidade,NaN,2022-04-19,Analise Visual Estrutural,NaN,NaN,NaN,0,2022
1,<NA>,2022,ABRIL,2022-04-19,NaN,NaN,"Avenida Dantas Barreto, `=-[",Santo Antonio,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0,2022
2,<NA>,2022,ABRIL,2022-04-19,NaN,NaN,"Rua Velha, ?_$.",Boa Vista,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0,2022


#### Tratamento das medidas numéricas

Durante o carregamento todas as medidas foram importadas como strings, inclusive as colunas originalmente numéricas.As colunas **quantidade**, **altura**, **largura** e **area_m** foram convertidas para o tipo numérico, e os valores nulos foram mantidos, já que essas medidas só se aplicam a parte dos atendimentos. Na coluna **area_m**, o valor 0 indica a ausência de medida, e não uma área nula, por isso é substituído por nulo, evitando distorções no cálculo de médias. Também foi criada a colunas de sinalização: **flag_area_divergente**, que identifica áreas diferentes do produto de altura, largura e quantidade.

In [8]:
print(df.dtypes)

regional         object
ano                 str
mes                 str
data_registro       str
ocorrencia          str
solicitacao         str
endereco            str
bairro              str
localidade          str
grau_risco          str
data_acao           str
tipo_acao           str
quantidade          str
altura              str
largura             str
area_m              str
ano_origem          str
dtype: object


In [9]:
for coluna in {"quantidade", "altura", "largura", "area_m"}:
    linhas_em_branco = df[f"{coluna}"].isna().sum()
    linhas_totais = len(df[f"{coluna}"])
    print(f'{coluna}: {linhas_em_branco} linhas em branco de {linhas_totais} no total ({(linhas_em_branco/linhas_totais)*100:.2f}%).')

largura: 220209 linhas em branco de 253762 no total (86.78%).
area_m: 0 linhas em branco de 253762 no total (0.00%).
quantidade: 215779 linhas em branco de 253762 no total (85.03%).
altura: 220193 linhas em branco de 253762 no total (86.77%).


In [10]:
MEDIDAS = ["quantidade", "altura", "largura", "area_m"]

def tratar_medidas(df):
    df = df.copy()

    for col in MEDIDAS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # tratamento das áreas nulas
    df["area_m"] = df["area_m"].where(df["area_m"] > 0)

    # tratamento das áreas calculadas erradas
    esperado = df["altura"] * df["largura"] * df["quantidade"]
    df["flag_area_divergente"] = (
        (esperado.notna() & ~((df["area_m"] - esperado).abs() <= 0.01))
        | (df["area_m"].notna() & esperado.isna())
    )
    return df

df = tratar_medidas(df)
assert len(df) == 253762

In [11]:
print(df.dtypes)

regional                 object
ano                         str
mes                         str
data_registro               str
ocorrencia                  str
solicitacao                 str
endereco                    str
bairro                      str
localidade                  str
grau_risco                  str
data_acao                   str
tipo_acao                   str
quantidade              float64
altura                  float64
largura                 float64
area_m                  float64
ano_origem                  str
flag_area_divergente       bool
dtype: object


#### Tratamento de linhas duplicadas

A base não possui um identificador único de atendimento, então não é possível saber diretamente se linhas idênticas são registros repetidos por engano ou atendimentos distintos com as mesmas características. Para avaliar o problema, são contadas as linhas totalmente iguais às anteriores dentro da base consolidada, por ano de origem.

In [12]:
colunas_base = [c for c in df.columns if c != "flag_area_divergente"]
dup = df.duplicated(subset=colunas_base, keep="first")

resumo = dup.groupby(df["ano_origem"]).agg(duplicatas="sum", total="size")
resumo["percentual"] = (resumo["duplicatas"] / resumo["total"] * 100).round(2)
resumo

,duplicatas,total,percentual
ano_origem,,,
2022,5735,90425,6.34
2023,5897,83164,7.09
2024,5931,80173,7.40


In [13]:
def remover_duplicatas(df):
    colunas_base = [c for c in df.columns if c != "flag_area_divergente"]
    return df.drop_duplicates(subset=colunas_base, keep="first").reset_index(drop=True)

antes = len(df)
df = remover_duplicatas(df)
print(f"{antes - len(df)} linhas removidas; {len(df)} restantes")

17563 linhas removidas; 236199 restantes


Foram identificadas 17.563 linhas duplicadas, correspondentes a 6,9% da base, distribuídas de forma semelhante entre os três anos (entre 6,3% e 7,4%). As duplicatas ocorrem apenas dentro de um mesmo arquivo e se concentram em ações porta a porta, que representam 53% delas, contra 16% da base. Como as linhas são idênticas em todas as colunas, inclusive data e endereço, assume-se que se tratam de registros repetidos, e as cópias são removidas, mantendo-se a primeira ocorrência.

#### Verificação especifica da coluna regional

Antes de Prosseguir é interresante checar quantas linhas estão com a coluna regional não preenchidas e quais são os diferentes tipos de dados presentes nela

In [14]:
print("\nQuantidade de regionais vazias:")
print(df["regional"].isna().sum())

print(df["regional"].value_counts(dropna=False))


Quantidade de regionais vazias:
84690
regional
<NA>        84690
Norte       34550
Noroeste    33815
Sul         27183
Nordeste    21220
Oeste       19623
Planície    14838
noroeste      279
NOroeste        1
Name: count, dtype: int64


#### Padronização Incial das Colunas

Como foi visto anteriormente a regional noroeste possuia 3 tipos de escritas diferentes, por isso é necessario padroniza-las, como futuramente vai ser necessario as colunas bairro e localidade para mapear as regionais que estão vazia, padroniza-las já as 3 se faz necessario

In [15]:
# padroniza as colunas de localização
for coluna in ["regional", "bairro", "localidade"]:

    df[coluna] = (
        df[coluna]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.title()
    )

# confere as regionais
print(df["regional"].value_counts(dropna=False))

regional
<NA>        84690
Norte       34550
Noroeste    34095
Sul         27183
Nordeste    21220
Oeste       19623
Planície    14838
Name: count, dtype: Int64


#### Verificação das colunas Bairro e Localidade

Da mesma forma que a coluna regional, é preciso checar quais são os tipos dos dados inseridos nessas duas colunas, para checar se existe formas diferentes de escrever a mesma informação

In [16]:
print("BAIRROS")
print("quantidade de bairros diferentes:", df["bairro"].nunique())
print("vazios:", df["bairro"].isna().sum())

# lista os nomes bairros existente
for bairro in sorted(df["bairro"].dropna().unique()):
    print(bairro)

print("\nLOCALIDADES")
print("quantidade de localidades diferentes:", df["localidade"].nunique())
print("vazios:", df["localidade"].isna().sum())

for localidade in sorted(df["localidade"].dropna().unique()):
    print(localidade)

BAIRROS
quantidade de bairros diferentes: 128
vazios: 498
Aflitos
Afogados
Agua Fria
Alto Da Bela Vista
Alto Do Eucalipto
Alto Do Mandu
Alto Do Maracana
Alto Jose Bonifacio
Alto Jose Do Pinho
Alto Santa Izabel
Alto Santa Terezinha
Apipucos
Areias
Arruda
Bairro Do Recife
Bairro Nao Cadastrado
Barro
Beberibe
Boa Viagem
Boa Vista
Bomba Do Hemeterio
Bongi
Brasilia Teimosa
Brejo Beberibe
Brejo Da Guabiraba
Brejo De Beberibe
Brejo Guabiraba
Cabanga
Cacote
Cajueiro
Campina Do Barreto
Campo Grande
Casa Amareal
Casa Amarela
Casa Forte
Caxanga
Chao De Estrelas
Cidade Operaria
Cidade Universitaria
Coelhos
Cohab
Coque
Coqueiral
Cordeiro
Corrego Da Bica
Corrego Da Fortuna
Corrego Da Padaria
Corrego Do Carroceiro
Corrego Do Fernandinho
Corrego Do Jenipapo
Corrego Do Joaquim
Corrego Do Marreco
Curado
Derby
Detran
Dois Irmaos
Dois Unidos
Encruzilhada
Engenho Do Meio
Espinheiro
Estancia
Fundao
Gracas
Guabiraba
Hipodromo
Ibura
Ibura Baixo
Ibura De Baixo
Ilha Do Leite
Ilha Do Retiro
Ilha Joana Bezerra
Im

Analisando essa lista foi possivel identificar que tem alguns nomes diferentes que significam a mesma coisa, por exemplo, o Bairro Casa Amarela tambem foi escrito como Casa Amareal, Os bairros Brejo tem algumas variações sem a particula de, entre outros, por isso é necessarios padroniza-los antes

#### Padronização das Colunas Bairro e Localidade

In [17]:
# corrige nomes equivalentes na coluna bairro
correcoes_bairro = {
    "Casa Amareal": "Casa Amarela",
    "Jd. Sao Paulo": "Jardim Sao Paulo",
    "Brejo Beberibe": "Brejo De Beberibe",
    "Brejo Guabiraba": "Brejo Da Guabiraba",
    "Ibura Baixo": "Ibura De Baixo",
    "Nao Informado" : pd.NA,
    "Sem Bairro" : pd.NA,
    "Bairro Nao Cadastrado" : pd.NA
}

df["bairro"] = df["bairro"].replace(correcoes_bairro)
df["bairro"] = df["bairro"].replace("Na", pd.NA)

# corrige nomes equivalentes na coluna localidade
correcoes_localidade = {
    "Altoda Bela Vista": "Alto Da Bela Vista",
    "Alto Bela Vista": "Alto Da Bela Vista",
    "Alto Do Cruzero": "Alto Do Cruzeiro",
    "Alto Resplendor": "Alto Do Resplendor",
    "Alto Dr. Caete": "Alto Dr. Caetes",
    "Vila Do Milagres": "Vila Dos Milagres",
    "T" : pd.NA,
    "Sem Localidade" : pd.NA,
    "Nao Cadastrada" : pd.NA,
    "Nao Informada" : pd.NA,
    "Localidade Nao Cadastrada" : pd.NA
}

df["localidade"] = df["localidade"].replace(correcoes_localidade)
df["localidade"] = df["localidade"].replace("Na", pd.NA)

# padroniza abreviações
df["localidade"] = (
    df["localidade"]
    .str.replace(r"^Cgo\.\s*", "Corrego ", regex=True)
    .str.replace(r"^Jd\.\s*", "Jardim ", regex=True)
)

df["bairro"] = (
    df["bairro"]
    .str.replace(r"^Jd\.\s*", "Jardim ", regex=True)
)

print("Bairros diferentes:", df["bairro"].nunique())
print("Localidades diferentes:", df["localidade"].nunique())

# lista os existentes apos as alterações
for bairro in sorted(df["bairro"].dropna().unique()):
    print(bairro)
for localidade in sorted(df["localidade"].dropna().unique()):
    print(localidade)

Bairros diferentes: 120
Localidades diferentes: 200
Aflitos
Afogados
Agua Fria
Alto Da Bela Vista
Alto Do Eucalipto
Alto Do Mandu
Alto Do Maracana
Alto Jose Bonifacio
Alto Jose Do Pinho
Alto Santa Izabel
Alto Santa Terezinha
Apipucos
Areias
Arruda
Bairro Do Recife
Barro
Beberibe
Boa Viagem
Boa Vista
Bomba Do Hemeterio
Bongi
Brasilia Teimosa
Brejo Da Guabiraba
Brejo De Beberibe
Cabanga
Cacote
Cajueiro
Campina Do Barreto
Campo Grande
Casa Amarela
Casa Forte
Caxanga
Chao De Estrelas
Cidade Operaria
Cidade Universitaria
Coelhos
Cohab
Coque
Coqueiral
Cordeiro
Corrego Da Bica
Corrego Da Fortuna
Corrego Da Padaria
Corrego Do Carroceiro
Corrego Do Fernandinho
Corrego Do Jenipapo
Corrego Do Joaquim
Corrego Do Marreco
Curado
Derby
Detran
Dois Irmaos
Dois Unidos
Encruzilhada
Engenho Do Meio
Espinheiro
Estancia
Fundao
Gracas
Guabiraba
Hipodromo
Ibura
Ibura De Baixo
Ilha Do Leite
Ilha Do Retiro
Ilha Joana Bezerra
Imbiribeira
Ipsep
Iputinga
Iraque
Jaqueira
Jardim Sao Paulo
Jiquia
Jordao
Lagoa Encant

Visto que não eram tantas as diferenças entres os nomes, criar um dicionario com as mudanças necessarias se tornou algo viavel, todas as diferenças de escritas para dizer que não possui informação sobre essa coluna foi substituida por NA, printando a nova lista é possivel ver que essas diferenças foram consertadas

#### Cruzamento de Colunas e Repopulação da Coluna Regional
Como a Tabela de 2022 não possui nenhuma informação relacionado a regional, foi acordado que o melhor jeito para prosseguir foi fazer o cruzamento entre bairro e localidade para achar a regional que mais aparece na combinação, assim repopulando as linhas cujo a regional estava vazia

In [19]:
# cria uma tabela com as tres colunas, que tenha as tres colunas não nulas
base_referencia = df.dropna(
    subset=["regional", "bairro", "localidade"]
)

# conta quantas vezes cada regional aparece por bairro e localidade
contagem = (
    base_referencia
    .groupby(["bairro", "localidade", "regional"])
    .size()
    .reset_index(name="quantidade")
)

# ordena pela maior quantidade, decrescente
contagem = contagem.sort_values(
    "quantidade",
    ascending=False
)

# mantem a regional mais frequente para cada bairro e localidade
mapa_regionais = contagem.drop_duplicates(
    subset=["bairro", "localidade"]
)

# cria uma correspondencia entre bairro, localidade e regional
correspondencias = mapa_regionais.set_index(
    ["bairro", "localidade"]
)["regional"] # serve como um dicionario que o index é o conjunto de bairro e localidade

# procura a regional correspondente para cada linha do df
regionais_encontradas = pd.Series(
    df.set_index(["bairro", "localidade"]).index.map(correspondencias),
    index=df.index,
    dtype="string" # procura no df original as linhas cujo o "index" se aplica
)

# preenche apenas as regionais que estao vazias
df["regional"] = df["regional"].fillna(regionais_encontradas)

# verifica quantas regionais continuam vazias
print("Regionais vazias:", df["regional"].isna().sum())

Regionais vazias: 9810


O print acima mostra que as linhas vazias da tabela regional diminuiram consideravelmente ou seja, o df foi repopulado consideravelmente

### Carregamento dos dados

#### Criação do schema do ETL

In [ ]:
with engine.begin() as conn:
    conn.execute(
        text("CREATE SCHEMA IF NOT EXISTS etl")
    )